# Online Retail II: Transaction Quality and Customer Sales EDA

### A business first exploration of more than one million retail transaction rows

This project started as a straightforward exploratory analysis. The interesting part quickly became data quality and transaction meaning.

Negative quantities are not automatically bad rows. Zero prices are not automatically missing values. Large quantities are not automatically outliers to delete. I use the transaction patterns and business context to decide what belongs in customer sales analysis.

**Questions I focus on**

* What does a clean sale look like in this dataset?
* Which negative quantities are customer cancellations and which look like internal stock adjustments?
* How much missing customer information is present?
* What do quantity and price distributions look like after transaction classification?
* How can the cleaned data support revenue, product, country and customer analysis?

## 1. Dataset

The project uses the UCI Online Retail II dataset, donated by Daqing Chen. It contains transactions for a UK based non store retailer from December 2009 to December 2011.

The workbook has two yearly sheets and 1,067,371 rows when combined.

Dataset reference: Chen, D. (2012), Online Retail II, UCI Machine Learning Repository, DOI 10.24432/C5CG6D.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidate_paths = [
    Path("data/online_retail_II.xlsx"),
    Path("online_retail_II.xlsx"),
    Path("/content/online_retail_II.xlsx")
]

file_path = next((path for path in candidate_paths if path.exists()), None)

if file_path is None:
    raise FileNotFoundError(
        "Download online_retail_II.xlsx from UCI and place it in the project folder or data folder before running the notebook."
    )

xls = pd.ExcelFile(file_path)
print("Sheets:", xls.sheet_names)

df_0910 = pd.read_excel(file_path, sheet_name="Year 2009-2010")
df_1011 = pd.read_excel(file_path, sheet_name="Year 2010-2011")

df_0910["source_year"] = "2009-2010"
df_1011["source_year"] = "2010-2011"

df = pd.concat([df_0910, df_1011], ignore_index=True)

print("Shape:", df.shape)
df.head()

Sheets: ['Year 2009-2010', 'Year 2010-2011']
Shape: (1067371, 9)


  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country source_year  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom   2009-2010  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom   2009-2010  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom   2009-2010  
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom   2009-2010  
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom   2009-2010  

## 2. First data quality pass

I start by checking missing values, duplicate rows, date coverage and the numeric ranges. The goal is not to clean immediately. First I want to understand why unusual values exist.

In [ ]:
quality_summary = pd.DataFrame({
    "missing_rows": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2)
})

display(quality_summary)

print("Exact duplicate rows:", df.duplicated().sum())
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

display(df[["Quantity", "Price"]].describe())

print("Negative quantity rows:", (df["Quantity"] < 0).sum())
print("Zero quantity rows:", (df["Quantity"] == 0).sum())
print("Negative price rows:", (df["Price"] < 0).sum())
print("Zero price rows:", (df["Price"] == 0).sum())

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
source_year         0
dtype: int64

Exact duplicate rows: 12133
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


           Quantity         Price
count  1.067371e+06  1.067371e+06
mean   9.938898e+00  4.649388e+00
std    1.727058e+02  1.235531e+02
min   -8.099500e+04 -5.359436e+04
25%    1.000000e+00  1.250000e+00
50%    3.000000e+00  2.100000e+00
75%    1.000000e+01  4.150000e+00
max    8.099500e+04  3.897000e+04

Negative quantity:  22950
Zero quantity:  0
Negative price:  5
Zero price:  6202


### Initial observations

The raw file contains 243,007 rows without a Customer ID and 12,133 exact duplicate rows.

There are 22,950 rows with negative quantity. That is far too many to label as random outliers without investigation. In transaction data, negative quantities often represent returns, cancellations or internal stock corrections.

There are also 6,202 zero price rows and 5 negative price rows. Those need separate treatment because they may represent operational records rather than normal customer sales.

## 3. Investigating negative quantities instead of deleting them

The source documentation says invoices beginning with `C` indicate cancellations. I use that rule first, then inspect negative quantity rows that do not have a cancellation invoice.

A useful lesson from the first draft of this notebook was to check the correct field. I initially tested the invoice date when I meant to test the invoice number. Fixing that changed the interpretation immediately.

In [ ]:
negative_quantity = df[df["Quantity"] < 0].copy()

print("First character of invoice for negative quantity rows:")
print(negative_quantity["Invoice"].astype(str).str[0].value_counts())

non_c_negative = df[
    (df["Quantity"] < 0)
    & (~df["Invoice"].astype(str).str.startswith("C"))
].copy()

print("Negative quantity rows without C invoice:", len(non_c_negative))
print("Rows with Customer ID:", non_c_negative["Customer ID"].notna().sum())

display(
    non_c_negative[
        ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]
    ].head(20)
)

non_c_negative["Description"].value_counts().head(20)

First character of invoice for negative quantity rows:
Invoice
C    19493
5     2728
4      729
Name: count, dtype: int64
Negative quantity rows without C invoice: 3457
Rows with Customer ID: 0


      Invoice StockCode      Description  Quantity  Price  Customer ID
263    489464     21733     85123a mixed       -96    0.0          NaN
283    489463     71477            short      -240    0.0          NaN
284    489467    85123A      21733 mixed      -192    0.0          NaN
470    489521     21646              NaN       -50    0.0          NaN
3114   489655     20683              NaN       -44    0.0          NaN
3162   489660     35956             lost     -1043    0.0          NaN
3168   489663    35605A          damages      -117    0.0          NaN
4296   489806     18010              NaN      -770    0.0          NaN
4538   489820     21133  invcd as 84879?      -720    0.0          NaN
4566   489821    85049G              NaN      -240    0.0          NaN
6556   489899   79323GR     sold as gold      -954    0.0          NaN
6576   489901     21098              NaN      -200    0.0          NaN
6911   490007     84347            21494      -720    0.0          NaN
7205  

Description
check                     123
damages                    84
?                          83
damaged                    78
missing                    27
sold as set on dotcom      20
Damaged                    17
smashed                     9
thrown away                 9
Unsaleable, destroyed.      9
dotcom                      8
??                          7
damages?                    7
crushed                     6
given away                  6
Damages                     5
wet damaged                 5
ebay                        5
counted                     5
MIA                         5
checked                     5
ebay sales                  4
Dotcom sales                4
CHECK                       3
broken                      3
Name: count, dtype: int64

The pattern is useful: all 3,457 negative quantity rows without a cancellation invoice are missing Customer ID, and common descriptions include `check`, `damages`, `missing`, `smashed` and similar operational language.

That makes them look much more like inventory or warehouse adjustments than customer purchases. I keep them in the raw data for auditability, but exclude them from normal customer sales analysis.

## 4. Price anomalies

The five negative prices are not ordinary product sales. In the raw data they are labelled `Adjust bad debt` and have no Customer ID.

Zero price rows are more mixed. Most are operational records, but 71 zero price rows do have a Customer ID. They may represent samples, replacements, test records or genuine customer interactions with no revenue. I flag them rather than silently converting them to paid sales.

In [ ]:
display(
    df[df["Price"] < 0][
        ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]
    ]
)

zero_price_customer = df[
    (df["Price"] == 0)
    & (df["Customer ID"].notna())
].copy()

print("Zero price rows with Customer ID:", len(zero_price_customer))
display(zero_price_customer.head(10))

        Invoice StockCode      Description  Quantity     Price  Customer ID
179403  A506401         B  Adjust bad debt         1 -53594.36          NaN
276274  A516228         B  Adjust bad debt         1 -44031.79          NaN
403472  A528059         B  Adjust bad debt         1 -38925.87          NaN
825444  A563186         B  Adjust bad debt         1 -11062.06          NaN
825445  A563187         B  Adjust bad debt         1 -11062.06          NaN

Zero price rows with Customer ID: 71


       Invoice StockCode                        Description  Quantity  Price  \
4674    489825     22076                 6 RIBBONS EMPIRE          12    0.0   
6781    489998     48185                DOOR MAT FAIRY CAKE         2    0.0   
16107   490727         M                             Manual         1    0.0   
18738   490961     22065     CHRISTMAS PUDDING TRINKET POT          1    0.0   
18739   490961     22142       CHRISTMAS CRAFT WHITE FAIRY         12    0.0   
32916   492079     85042          ANTIQUE LILY FAIRY LIGHTS         8    0.0   
40101   492760     21143    ANTIQUE GLASS HEART DECORATION         12    0.0   
47126   493761     79320                    FLAMINGO LIGHTS        24    0.0   
48342   493899     22355        CHARLOTTE BAG , SUKI DESIGN        10    0.0   
57619   494607     21533          RETRO SPOT LARGE MILK JUG        12    0.0   
89084   497819   TEST001            This is a test product.         5    0.0   
89180   497843   TEST001            This

## 5. Build an analysis ready transaction table

I keep the original dataframe unchanged and create a cleaned copy for analysis.

Exact duplicates are removed first. Then each remaining row is classified into one of five transaction types:

* cancellation
* inventory adjustment
* accounting adjustment
* zero price record
* normal sale

The order of these rules matters. A cancellation should stay a cancellation even if other fields are unusual.

In [ ]:
clean_df = df.drop_duplicates().copy()

clean_df["is_cancellation"] = (
    clean_df["Invoice"].astype(str).str.startswith("C")
)

clean_df["is_inventory_adjustment"] = (
    (clean_df["Quantity"] < 0)
    & (~clean_df["is_cancellation"])
)

clean_df["is_negative_price"] = clean_df["Price"] < 0
clean_df["is_zero_price"] = clean_df["Price"] == 0

conditions = [
    clean_df["is_cancellation"],
    clean_df["is_inventory_adjustment"],
    clean_df["is_negative_price"],
    clean_df["is_zero_price"]
]

choices = [
    "cancellation",
    "inventory_adjustment",
    "accounting_adjustment",
    "zero_price"
]

clean_df["transaction_type"] = np.select(
    conditions,
    choices,
    default="normal_sale"
)

transaction_counts = clean_df["transaction_type"].value_counts()
transaction_counts

For revenue and customer behaviour analysis, I use only `normal_sale` rows. This prevents returns, bad debt adjustments and free or operational records from inflating purchase metrics.

In [ ]:
sales_df = clean_df[
    clean_df["transaction_type"] == "normal_sale"
].copy()

sales_df["revenue"] = sales_df["Quantity"] * sales_df["Price"]

sales_summary = pd.Series({
    "sales_rows": len(sales_df),
    "sales_invoices": sales_df["Invoice"].nunique(),
    "identified_customers": sales_df["Customer ID"].nunique(),
    "countries": sales_df["Country"].nunique(),
    "revenue": sales_df["revenue"].sum()
})

sales_summary

## 6. Quantity and price distributions

Retail transaction data is heavily skewed. I do not remove high quantity rows just because they are far from the median. Some customers are wholesalers, and some very large purchases are followed by matching cancellations.

I inspect quantiles and extreme rows first, then use a trimmed view only for visualization.

In [ ]:
quantity_quantiles = sales_df["Quantity"].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999]
)

price_quantiles = sales_df["Price"].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999]
)

print("Quantity quantiles")
display(quantity_quantiles)

print("Price quantiles")
display(price_quantiles)

quantity_cutoff = sales_df["Quantity"].quantile(0.99)
price_cutoff = sales_df["Price"].quantile(0.99)

plt.figure(figsize=(9, 5))
plt.hist(
    sales_df.loc[sales_df["Quantity"] <= quantity_cutoff, "Quantity"],
    bins=50
)
plt.xlabel("Quantity")
plt.ylabel("Line items")
plt.title("Quantity distribution up to the 99th percentile")
plt.show()

plt.figure(figsize=(9, 5))
plt.hist(
    sales_df.loc[sales_df["Price"] <= price_cutoff, "Price"],
    bins=50
)
plt.xlabel("Unit price")
plt.ylabel("Line items")
plt.title("Price distribution up to the 99th percentile")
plt.show()

From the raw exploration, quantity is strongly right skewed. The median is 3 units and the 75th percentile is 10 units. A small number of very large transactions stretch the distribution considerably.

That is a good example of why I prefer investigation before automatic outlier removal. In business data, an extreme value can be a large wholesale order, a reversal, or a data issue. Those cases have different meanings even if they look similar statistically.

## 7. Revenue trend

After defining normal sales, monthly revenue becomes much more meaningful because cancellations and operational adjustments are no longer mixed into the same measure.

In [ ]:
sales_df["month"] = sales_df["InvoiceDate"].dt.to_period("M").dt.to_timestamp()

monthly_revenue = (
    sales_df.groupby("month", as_index=False)["revenue"]
    .sum()
)

plt.figure(figsize=(11, 5))
plt.plot(monthly_revenue["month"], monthly_revenue["revenue"], marker="o")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.title("Monthly revenue from normal sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

monthly_revenue.tail()

## 8. Product and country analysis

For product ranking I use revenue rather than only units sold, because high volume low price products and lower volume high price products tell different stories.

For country analysis I show both the United Kingdom and non UK markets so the home market does not hide the international pattern.

In [ ]:
product_performance = (
    sales_df.dropna(subset=["Description"])
    .groupby("Description", as_index=False)
    .agg(
        units_sold=("Quantity", "sum"),
        revenue=("revenue", "sum"),
        invoices=("Invoice", "nunique")
    )
    .sort_values("revenue", ascending=False)
)

display(product_performance.head(15))

country_performance = (
    sales_df.groupby("Country", as_index=False)
    .agg(
        revenue=("revenue", "sum"),
        invoices=("Invoice", "nunique"),
        customers=("Customer ID", "nunique")
    )
    .sort_values("revenue", ascending=False)
)

print("Top countries overall")
display(country_performance.head(10))

print("Top countries outside the United Kingdom")
display(
    country_performance[
        country_performance["Country"] != "United Kingdom"
    ].head(10)
)

## 9. Customer level view

Customer analysis requires a known Customer ID, so unidentified sales stay in the revenue analysis but are excluded from customer metrics.

I aggregate orders, revenue, units, average order value and purchase dates. This creates a clean customer table that can later support retention analysis, RFM work or predictive modelling.

In [ ]:
customer_sales = sales_df.dropna(subset=["Customer ID"]).copy()

invoice_totals = (
    customer_sales.groupby(["Customer ID", "Invoice"], as_index=False)
    .agg(
        invoice_date=("InvoiceDate", "max"),
        order_value=("revenue", "sum"),
        units=("Quantity", "sum")
    )
)

customer_summary = (
    invoice_totals.groupby("Customer ID", as_index=False)
    .agg(
        orders=("Invoice", "nunique"),
        total_revenue=("order_value", "sum"),
        avg_order_value=("order_value", "mean"),
        total_units=("units", "sum"),
        first_purchase=("invoice_date", "min"),
        last_purchase=("invoice_date", "max")
    )
)

snapshot_date = sales_df["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)
customer_summary["recency_days"] = (
    snapshot_date - customer_summary["last_purchase"].dt.normalize()
).dt.days

customer_summary.sort_values("total_revenue", ascending=False).head(10)

## 10. Cancellation view

Cancellations are not included in sales revenue, but they are still business events worth measuring. A high cancellation rate can point to product, fulfilment or customer experience issues.

I calculate cancellation value using the absolute quantity so the magnitude is easier to interpret.

In [ ]:
cancellations = clean_df[
    clean_df["transaction_type"] == "cancellation"
].copy()

cancellations["cancelled_value"] = (
    cancellations["Quantity"].abs() * cancellations["Price"].clip(lower=0)
)

cancellation_summary = pd.Series({
    "cancellation_rows": len(cancellations),
    "cancelled_invoices": cancellations["Invoice"].nunique(),
    "customers_with_cancellation": cancellations["Customer ID"].nunique(),
    "cancelled_value": cancellations["cancelled_value"].sum()
})

cancellation_summary

## 11. Key findings from the investigation

* The raw workbook contains 1,067,371 transaction rows across two yearly sheets.
* 243,007 rows have no Customer ID, so they can contribute to revenue analysis but not reliable customer level analysis.
* There are 12,133 exact duplicate rows, which should be removed before aggregation.
* 22,950 rows have negative quantity. Of these, 19,493 use the cancellation invoice pattern and 3,457 do not.
* The 3,457 negative quantity rows without a cancellation invoice have no Customer ID and frequently use descriptions such as `damages`, `check`, `missing` and related operational terms. I treat them as inventory adjustments.
* The five negative price rows are bad debt adjustments rather than product sales.
* 6,202 rows have zero price. Only 71 of those have a Customer ID, so I keep them separate from normal paid sales.
* Quantity is strongly right skewed. The median raw quantity is 3 and the 75th percentile is 10, while a small number of very large transactions create a long right tail.

The most useful part of this project was not a chart or a single KPI. It was learning to separate statistical oddities from business events. Deleting every negative or extreme value would have produced a cleaner looking dataset and a worse analysis.

## 12. Where I would take the project next

This notebook creates a reliable analytical base rather than jumping straight to a predictive model.

Possible next steps are:

* cohort retention analysis using first purchase month
* RFM segmentation on identified customers
* repeat purchase prediction using an observation and target window
* cancellation analysis by product and customer segment
* a Power BI dashboard built from the cleaned transaction and customer tables

For any predictive extension, I would keep the same principle used here: define the business event first, then build the feature set around that definition.